# Lecture 03: every figure in the slides

These cells are the code that produced the linear regression figures in the deck, with each file write replaced by an inline plot. The data file sits beside this notebook in the course repository; the first cell fetches it when the notebook is opened in Colab.

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The course palette and figure sizes, copied from slides/figures/sds265_style.py
BLUE, BLUE_BRIGHT, RED, GREEN = '#00356B', '#286DC0', '#7A3E47', '#5B7864'
INK, MUTED, RULE, BLUE_PALE = '#202A35', '#66727E', '#D8E0E8', '#F2F6FA'
BLUE_FILL, RED_FILL, GREEN_FILL = '#8C9FB1', '#B0846F', '#9AAE96'
MORANDI_BLUE, MORANDI_CLAY, MORANDI_SAGE = '#8C9FB1', '#B0846F', '#9AAE96'
MORANDI_MAUVE, MORANDI_SAND, MORANDI_STONE = '#A98FA0', '#C7B392', '#8A8F8A'
MORANDI = [MORANDI_BLUE, MORANDI_CLAY, MORANDI_SAGE, MORANDI_MAUVE, MORANDI_SAND, MORANDI_STONE]
MORANDI_EDGE = {MORANDI_BLUE: '#5C7186', MORANDI_CLAY: '#835747', MORANDI_SAGE: '#6B8067',
                MORANDI_MAUVE: '#7C6373', MORANDI_SAND: '#9A8763', MORANDI_STONE: '#5F645F'}
CYCLE = MORANDI
TEXTWIDTH, TEXTHEIGHT = 5.63, 3.09
FULL, FULL_TALL, FULL_SHORT = (TEXTWIDTH, 2.0), (TEXTWIDTH, 2.35), (TEXTWIDTH, 1.7)
HALF, HALF_TALL = (2.70, 2.05), (2.70, 2.45)

def use_course_style():
    mpl.rcParams.update({'figure.dpi': 130, 'font.size': 9, 'axes.titlecolor': BLUE,
                         'axes.edgecolor': MUTED, 'axes.linewidth': .7,
                         'axes.spines.top': False, 'axes.spines.right': False,
                         'xtick.color': MUTED, 'ytick.color': MUTED,
                         'legend.frameon': False, 'lines.linewidth': 1.8,
                         'axes.prop_cycle': mpl.cycler(color=CYCLE)})

def save(fig, *paths):     # the deck writes files; the notebook shows the plot
    plt.show()
    return Path(paths[0]) if paths else Path('figure.pdf')

use_course_style()
# The deck scripts derive output paths from OUT and NOTES. Nothing is written
# here, so both point at the working directory and the notebook runs anywhere.
OUT = Path('.')
NOTES = Path('.')

In [ ]:
from __future__ import annotations


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


_COST: dict = {}

DATA = Path("Advertising.csv")
#: The Quarto lecture notes render to HTML, so they need SVG copies.


def notes_svg(name: str) -> Path:
    """Path of the notes copy of a figure, using the chapter naming scheme."""
    return NOTES / f"ch03-{name}.svg"

ad = pd.read_csv(DATA, index_col=0)
tv, radio, news, sales = ad.TV.values, ad.radio.values, ad.newspaper.values, ad.sales.values


def ols(X, y):
    """Least squares by QR, returning coefficients, standard errors, residuals."""
    X = np.asarray(X, float)
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    n, p = X.shape
    sigma2 = resid @ resid / (n - p)
    cov = sigma2 * np.linalg.inv(X.T @ X)
    return beta, np.sqrt(np.diag(cov)), resid, sigma2


ones = np.ones_like(tv)
beta_tv, se_tv, res_tv, s2_tv = ols(np.c_[ones, tv], sales)
beta_all, se_all, res_all, s2_all = ols(np.c_[ones, tv, radio, news], sales)


# --- 1a. The raw scatter, before any model has been introduced --------------
def fig_panels_raw():
    """The opening look at the data.

    Deliberately has no fitted line: at this point in the lecture neither the
    linear model nor least squares has been introduced, so a line would be
    unexplained. It appears once it has been earned.
    """
    fig, axes = plt.subplots(1, 3, figsize=FULL_SHORT, sharey=True)
    for ax, (x, name) in zip(axes, [(tv, "TV"), (radio, "radio"), (news, "newspaper")]):
        ax.scatter(x, sales, s=9, color=MORANDI_BLUE, alpha=0.55, linewidths=0)
        ax.set_xlabel(f"{name} budget")
    axes[0].set_ylabel("sales")
    fig.tight_layout(pad=0.3)
    return save(fig, OUT / "advertising-panels-raw.pdf", notes_svg("advertising-panels-raw"))

fig_panels_raw()

In [ ]:
# --- 1b. The same panels once a line can be justified -----------------------
def fig_panels():
    fig, axes = plt.subplots(1, 3, figsize=FULL_SHORT, sharey=True)
    for ax, (x, name) in zip(axes, [(tv, "TV"), (radio, "radio"), (news, "newspaper")]):
        ax.scatter(x, sales, s=9, color=MORANDI_BLUE, alpha=0.55, linewidths=0)
        b, *_ = ols(np.c_[np.ones_like(x), x], sales)
        grid = np.linspace(x.min(), x.max(), 2)
        ax.plot(grid, b[0] + b[1] * grid, color=MORANDI_EDGE[MORANDI_CLAY], lw=1.9)
        ax.set_xlabel(f"{name} budget")
        ax.set_title(f"slope {b[1]:.4f}")
    axes[0].set_ylabel("sales")
    fig.tight_layout(pad=0.3)
    return save(fig, OUT / "advertising-panels.pdf", notes_svg("advertising-panels"))

fig_panels()

In [ ]:
# --- 2. ISLP Fig. 3.1 recreation: the fit and its vertical residuals --------
def fig_tv_fit():
    fig, ax = plt.subplots(figsize=FULL)
    order = np.argsort(tv)
    fit = beta_tv[0] + beta_tv[1] * tv
    for xi, yi, fi in zip(tv, sales, fit):
        ax.plot([xi, xi], [fi, yi], color=MUTED, lw=0.55, alpha=0.85, zorder=1)
    ax.scatter(tv, sales, s=13, color=MORANDI_CLAY, alpha=0.85, linewidths=0, zorder=3)
    ax.plot(tv[order], fit[order], color=MORANDI_EDGE[MORANDI_BLUE], lw=2.1, zorder=4)
    ax.set_xlabel("TV advertising budget  (thousands of dollars)")
    ax.set_ylabel("sales  (thousands of units)")
    ax.annotate(
        rf"$\widehat{{y}} = {beta_tv[0]:.3f} + {beta_tv[1]:.4f}\,x$",
        xy=(0.035, 0.93),
        xycoords="axes fraction",
        color=BLUE,
        fontsize=10,
    )
    ax.annotate(
        r"each grey segment is a residual $r_i = y_i - \widehat{y}_i$",
        xy=(0.035, 0.845),
        xycoords="axes fraction",
        color=MUTED,
        fontsize=8.5,
    )
    fig.tight_layout(pad=0.3)
    return save(fig, OUT / "advertising-tv-fit.pdf", notes_svg("advertising-tv-fit"))

fig_tv_fit()

In [ ]:
# --- 2b. Residuals measured against a line we have NOT yet optimized -------
def fig_candidate_line():
    """Residuals to an arbitrary candidate line.

    Used where residuals and RSS are defined, which is before least squares has
    been derived. Showing the fitted line there would quote coefficients the
    lecture has not yet computed; a deliberately suboptimal line makes the point
    that RSS is a function of whichever line you pick.
    """
    b0, b1 = 4.0, 0.075          # a plausible but clearly suboptimal guess
    fit = b0 + b1 * tv
    rss = float(((sales - fit) ** 2).sum())

    fig, ax = plt.subplots(figsize=FULL)
    order = np.argsort(tv)
    for xi, yi, fi in zip(tv, sales, fit):
        ax.plot([xi, xi], [fi, yi], color=MUTED, lw=0.55, alpha=0.85, zorder=1)
    ax.scatter(tv, sales, s=13, color=MORANDI_CLAY, alpha=0.85, linewidths=0, zorder=3)
    ax.plot(tv[order], fit[order], color=MORANDI_EDGE[MORANDI_BLUE], lw=2.1, zorder=4)
    ax.set_xlabel("TV advertising budget  (thousands of dollars)")
    ax.set_ylabel("sales  (thousands of units)")
    ax.annotate(rf"a candidate line:  $\beta_0={b0:.0f},\ \beta_1={b1:.3f}$",
                xy=(0.035, 0.93), xycoords="axes fraction", color=BLUE, fontsize=9.5)
    ax.annotate(rf"RSS $= {rss:,.0f}$",
                xy=(0.035, 0.845), xycoords="axes fraction", color=MUTED, fontsize=9)
    fig.tight_layout(pad=0.3)
    return save(fig, OUT / "candidate-line.pdf", notes_svg("candidate-line"))

fig_candidate_line()

In [ ]:
# --- 3. ISLP Fig. 3.2 recreation: the RSS surface over (beta0, beta1) -------
def fig_rss_surface():
    b0 = np.linspace(5.0, 9.2, 150)
    b1 = np.linspace(0.030, 0.065, 150)
    B0, B1 = np.meshgrid(b0, b1)
    resid = sales[:, None, None] - B0[None] - B1[None] * tv[:, None, None]
    RSS = (resid**2).mean(axis=0)          # the empirical risk, not the sum

    fig = plt.figure(figsize=FULL)
    ax1 = fig.add_subplot(1, 2, 1)
    levels = np.array([10.6, 10.8, 11.0, 11.5, 12.5, 14.0, 16.0, 19.0, 23.0])
    cs = ax1.contour(B0, B1, RSS, levels=levels, colors=BLUE_BRIGHT, linewidths=0.7)
    ax1.clabel(cs, inline=True, fontsize=7.5, fmt="%.0f")
    ax1.plot(beta_tv[0], beta_tv[1], marker="*", ms=13, color=MORANDI_EDGE[MORANDI_CLAY], zorder=5)
    ax1.set_xlabel(r"$\beta_0$")
    ax1.set_ylabel(r"$\beta_1$")
    ax1.set_title(r"contours of $\widehat{R}_n$")

    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    ax2.plot_surface(
        B0, B1, RSS, cmap="Blues_r", linewidth=0, antialiased=True,
        alpha=0.95, rstride=4, cstride=4,
    )
    # Quiet the default 3D furniture so the bowl itself is the only message.
    for pane_axis in (ax2.xaxis, ax2.yaxis, ax2.zaxis):
        pane_axis.pane.set_alpha(0.0)
        pane_axis.pane.set_edgecolor("none")
        pane_axis._axinfo["grid"].update(color=RULE, linewidth=0.4)
    # The contour panel carries the numbers; the surface only has to show the
    # shape, so its tick labels are removed rather than shrunk to illegibility.
    ax2.set_xlabel(r"$\beta_0$", labelpad=-12, color=INK)
    ax2.set_ylabel(r"$\beta_1$", labelpad=-12, color=INK)
    ax2.set_xticklabels([])
    ax2.set_yticklabels([])
    ax2.set_zticks([])
    ax2.tick_params(length=0)
    ax2.view_init(elev=26, azim=-128)
    ax2.set_box_aspect((1, 1, 0.72))
    ax2.set_title("the same risk as a surface")
    fig.tight_layout(pad=0.25)
    return save(fig, OUT / "rss-surface.pdf", notes_svg("rss-surface"))

fig_rss_surface()

In [ ]:
# --- 4. Why the newspaper coefficient dies in the multiple fit --------------
def fig_confounding():
    """A purely predictive account of the vanishing newspaper effect.

    Left: newspaper and radio budgets move together. Right: strip out the part
    of newspaper that radio already predicts, and what is left has essentially
    no relationship with sales. No inference, no p-values.
    """
    fig, axes = plt.subplots(1, 2, figsize=FULL)

    ax = axes[0]
    ax.scatter(radio, news, s=11, color=MORANDI_BLUE, alpha=0.6, linewidths=0)
    b, *_ = ols(np.c_[np.ones_like(radio), radio], news)
    grid = np.linspace(radio.min(), radio.max(), 2)
    ax.plot(grid, b[0] + b[1] * grid, color=MORANDI_EDGE[MORANDI_CLAY], lw=1.7)
    ax.set_xlabel("radio budget")
    ax.set_ylabel("newspaper budget")
    ax.set_title(f"the two move together (corr {np.corrcoef(radio, news)[0,1]:.2f})",
                 fontsize=8.4)

    # The part of newspaper that radio cannot predict.
    news_res = news - (b[0] + b[1] * radio)
    ax = axes[1]
    ax.scatter(news_res, sales, s=11, color=MORANDI_BLUE, alpha=0.6, linewidths=0)
    c, *_ = ols(np.c_[np.ones_like(news_res), news_res], sales)
    g2 = np.linspace(news_res.min(), news_res.max(), 2)
    ax.plot(g2, c[0] + c[1] * g2, color=MORANDI_EDGE[MORANDI_CLAY], lw=1.9)
    ax.set_xlabel(r"$\widetilde{x}$ = newspaper $-$ ($\widehat{\gamma}_0+\widehat{\gamma}_1\,$radio)")
    ax.set_ylabel("sales")
    ax.set_title(f"and what is left explains nothing (slope {c[1]:.4f})", fontsize=8.4)
    fig.tight_layout(pad=0.4)
    return save(fig, OUT / "newspaper-confounding.pdf", notes_svg("newspaper-confounding"))

fig_confounding()

In [ ]:
# --- 5. Residual diagnostics: the TV-only fit is visibly wrong --------------
# UNUSED: no deck includes residual-diagnostics.pdf. Kept in case a slide wants it back.
def fig_diagnostics():
    fig, axes = plt.subplots(1, 2, figsize=FULL)
    fit_tv = beta_tv[0] + beta_tv[1] * tv
    fit_all = beta_all[0] + beta_all[1] * tv + beta_all[2] * radio + beta_all[3] * news

    panels = [
        (fit_tv, res_tv, r"sales $\sim$ TV"),
        (fit_all, res_all, r"sales $\sim$ TV + radio + newspaper"),
    ]
    for ax, (f, r, title) in zip(axes, panels):
        ax.scatter(f, r, s=11, color=MORANDI_BLUE, alpha=0.6, linewidths=0)
        ax.axhline(0.0, color=MUTED, lw=0.9, ls=(0, (4, 3)))
        # A quadratic fit to the residuals is enough to expose the curvature
        # students should see; no extra dependency needed.
        q = np.polyfit(f, r, 2)
        grid = np.linspace(f.min(), f.max(), 120)
        ax.plot(grid, np.polyval(q, grid), color=MORANDI_EDGE[MORANDI_CLAY], lw=1.8)
        ax.set_xlabel(r"fitted value $\widehat{y}$")
        ax.set_title(title)
    axes[0].set_ylabel("residual")
    fig.tight_layout(pad=0.35)
    return save(fig, OUT / "residual-diagnostics.pdf", notes_svg("residual-diagnostics"))

fig_diagnostics()

In [ ]:
# --- 6. Leverage and influence ---------------------------------------------
# UNUSED: no deck includes leverage-influence.pdf. Kept in case a slide wants it back.
def fig_leverage():
    X = np.c_[ones, tv, radio]
    _, _, res, s2 = ols(X, sales)
    H = X @ np.linalg.solve(X.T @ X, X.T)
    h = np.diag(H)
    stud = res / np.sqrt(s2 * (1.0 - h))
    n, p = X.shape

    fig, ax = plt.subplots(figsize=FULL)
    ax.scatter(h, stud, s=14, color=MORANDI_BLUE, alpha=0.6, linewidths=0)
    ax.axhline(0.0, color=RULE, lw=1.0)
    ax.axvline(p / n, color=MUTED, lw=0.9, ls=(0, (4, 3)))
    top = ax.get_ylim()[1]
    ax.annotate(
        r"average leverage $p/n$",
        xy=(p / n, top),
        xytext=(5, -11),
        textcoords="offset points",
        color=MUTED,
        fontsize=8,
    )
    worst = int(np.argmax(h))
    ax.scatter([h[worst]], [stud[worst]], s=55, facecolor="none", edgecolor=MORANDI_EDGE[MORANDI_CLAY], lw=1.5)
    ax.annotate(
        f"market {ad.index[worst]}",
        xy=(h[worst], stud[worst]),
        xytext=(-52, 14),
        textcoords="offset points",
        color=RED,
        fontsize=8.5,
        arrowprops=dict(arrowstyle="-", color=RED, lw=0.8),
    )
    ax.set_xlabel(r"leverage $h_i = (H)_{ii}$")
    ax.set_ylabel("studentized residual")
    fig.tight_layout(pad=0.3)
    return save(fig, OUT / "leverage-influence.pdf", notes_svg("leverage-influence"))

fig_leverage()

In [ ]:
# --- 7. Squared versus absolute loss ---------------------------------------
def fig_loss_shapes():
    """Squared, absolute, and Huber loss.

    Huber is quadratic near zero, so it keeps a unique smooth minimum, and
    linear in the tails, so a single wild observation cannot dominate.
    """
    r = np.linspace(-3, 3, 600)
    delta = 1.0
    huber = np.where(np.abs(r) <= delta, 0.5 * r**2, delta * (np.abs(r) - 0.5 * delta))
    fig, ax = plt.subplots(figsize=HALF_TALL)
    ax.plot(r, 0.5 * r**2, color=MORANDI_EDGE[MORANDI_BLUE], lw=2.0, label=r"squared  $\frac{1}{2}r^2$")
    ax.plot(r, np.abs(r), color=MORANDI_EDGE[MORANDI_CLAY], lw=2.0, label=r"absolute  $|r|$")
    ax.plot(r, huber, color=MORANDI_EDGE[MORANDI_SAGE], lw=2.0, ls=(0, (5, 2)),
            label=rf"Huber  $\delta={delta:.0f}$")
    ax.axvline(-delta, color=RULE, lw=0.9)
    ax.axvline(delta, color=RULE, lw=0.9)
    ax.set_xlabel(r"residual $r$")
    ax.set_ylabel("loss")
    ax.legend(loc="upper center", fontsize=8.0)
    fig.tight_layout(pad=0.3)
    return save(fig, OUT / "loss-shapes.pdf", notes_svg("loss-shapes"))

fig_loss_shapes()

In [ ]:
# --- 8. Gradient descent on the Advertising least-squares problem ----------


# --- 9. The error decomposition: optimization versus statistical error ------


def report():
    """Print every number quoted on the slides so it can be checked."""
    tss = float(((sales - sales.mean()) ** 2).sum())
    print(f"simple   sales ~ TV : b0={beta_tv[0]:.4f} (se {se_tv[0]:.4f}), "
          f"b1={beta_tv[1]:.4f} (se {se_tv[1]:.4f})")
    print(f"           t = {beta_tv[0]/se_tv[0]:.2f}, {beta_tv[1]/se_tv[1]:.2f}; "
          f"RSE={np.sqrt(s2_tv):.3f}; R^2={1 - res_tv @ res_tv / tss:.4f}; "
          f"RSS={res_tv @ res_tv:.1f}; TSS={tss:.1f}")
    print("multiple sales ~ TV + radio + newspaper:")
    for lab, b, s in zip(["intercept", "TV", "radio", "newspaper"], beta_all, se_all):
        print(f"           {lab:10s} {b:9.4f}  se {s:.4f}  t {b / s:7.2f}")
    print(f"           RSE={np.sqrt(s2_all):.3f}, R^2={1 - res_all @ res_all / tss:.4f}")
    print(f"           corr(radio, newspaper) = {np.corrcoef(radio, news)[0, 1]:.4f}")
    if _COST:
        print(f"timing  n={_COST['n']:,}")
        for p_, e, g in zip(_COST["ps"], _COST["exact"], _COST["grad"]):
            print(f"           p={p_:5d}  exact {e*1000:9.2f} ms   one GD step {g*1000:7.3f} ms"
                  f"   ratio {e/g:8.1f}")
    xbar, ybar = tv.mean(), sales.mean()
    sxy = float(((tv - xbar) * (sales - ybar)).sum())
    sxx = float(((tv - xbar) ** 2).sum())
    print(f"worked   xbar={xbar:.3f} ybar={ybar:.3f} Sxy={sxy:.1f} Sxx={sxx:.1f} "
          f"-> b1={sxy / sxx:.6f}, b0={ybar - sxy / sxx * xbar:.4f}")




# --- 10. Convexity: the tangent-line property and what uniqueness needs -----
def fig_convexity():
    """Nonconvex, convex-but-not-strictly, and strictly convex, side by side."""
    fig, axes = plt.subplots(1, 3, figsize=FULL)
    t = np.linspace(-2.2, 2.2, 600)

    # (a) not convex: a chord can pass below the graph
    nc = 0.42 * t**4 - 1.35 * t**2 + 1.4
    ax = axes[0]
    ax.plot(t, nc, color=MORANDI_EDGE[MORANDI_BLUE], lw=2.2)
    a, b = -1.55, 1.55
    fa = 0.42 * a**4 - 1.35 * a**2 + 1.4
    fb = 0.42 * b**4 - 1.35 * b**2 + 1.4
    ax.plot([a, b], [fa, fb], color=MORANDI_EDGE[MORANDI_CLAY], lw=1.6, ls=(0, (5, 3)))
    ax.plot([a, b], [fa, fb], "o", ms=5, color=MORANDI_EDGE[MORANDI_CLAY])
    ax.annotate("the chord dips\nbelow the graph", xy=(0, 0.5 * (fa + fb)),
                xytext=(0, 16), textcoords="offset points", ha="center",
                color=RED, fontsize=7.8)
    ax.set_title("not convex", fontsize=9)

    # (b) convex but not strictly: a flat stretch of minimizers
    g = np.where(np.abs(t) <= 0.9, 0.4, 0.62 * (np.abs(t) - 0.9) ** 2 + 0.4)
    ax = axes[1]
    ax.plot(t, g, color=MORANDI_EDGE[MORANDI_BLUE], lw=2.2)
    ax.plot([-0.9, 0.9], [0.4, 0.4], color=MORANDI_EDGE[MORANDI_CLAY], lw=4.5, alpha=0.5,
            solid_capstyle="butt")
    ax.annotate("every point here\nis a minimizer", xy=(0, 0.4), xytext=(0, 20),
                textcoords="offset points", ha="center", color=RED, fontsize=7.8)
    ax.set_title("convex, not strictly", fontsize=9)

    # (c) strictly convex: exactly one minimizer
    sc = 0.62 * t**2 + 0.4
    ax = axes[2]
    ax.plot(t, sc, color=MORANDI_EDGE[MORANDI_BLUE], lw=2.2)
    ax.plot([0], [0.4], "*", ms=14, color=MORANDI_EDGE[MORANDI_SAGE])
    ax.annotate("exactly one\nminimizer", xy=(0, 0.4), xytext=(0, 20),
                textcoords="offset points", ha="center", color=GREEN, fontsize=7.8)
    ax.set_title("strictly convex", fontsize=9)

    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(r"$\beta$")
        ax.set_ylim(-0.5, 3.6)
    fig.tight_layout(pad=0.4)
    return save(fig, OUT / "convexity.pdf", notes_svg("convexity"))

fig_convexity()

In [ ]:
# --- 10d. A stationary point of a convex function is a global minimum -------
def fig_stationary_global():
    """At a flat point the tangent is horizontal, and the graph stays above it."""
    t = np.linspace(-2.4, 2.4, 500)
    f = 0.6 * t**2 + 0.5
    fig, ax = plt.subplots(figsize=HALF_TALL)
    ax.plot(t, f, color=MORANDI_EDGE[MORANDI_BLUE], lw=2.4, zorder=3)
    ax.axhline(0.5, color=MORANDI_EDGE[MORANDI_CLAY], lw=1.7, ls=(0, (5, 3)), zorder=2)
    ax.fill_between(t, 0.5, f, color=MORANDI_BLUE, alpha=0.10, zorder=1)
    ax.plot([0], [0.5], "o", ms=6, color=MORANDI_EDGE[MORANDI_CLAY], zorder=4)
    ax.annotate(r"$\nabla L(\beta^{\circ})=0$", xy=(0, 0.5), xytext=(34, -6),
                textcoords="offset points", ha="left", color=RED, fontsize=8.4)
    ax.annotate("the graph never goes\nbelow this level", xy=(0, 3.15),
                xytext=(0, 0), textcoords="offset points", ha="center",
                color=BLUE, fontsize=8.0)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(r"$\beta$")
    ax.set_ylim(0.05, 4.4)
    fig.tight_layout(pad=0.3)
    return save(fig, OUT / "stationary-global.pdf", notes_svg("stationary-global"))

fig_stationary_global()

In [ ]:
# --- 11. Collinearity: the RSS valley and unstable coefficients -------------
def _collinear_design(rho, n=200, seed=0):
    rng = np.random.default_rng(seed)
    z = rng.normal(size=n)
    x1 = z
    x2 = rho * z + np.sqrt(max(1e-12, 1 - rho**2)) * rng.normal(size=n)
    X = np.column_stack([x1, x2])
    X = (X - X.mean(0)) / X.std(0)
    return X


# UNUSED: no deck includes collinearity-contours.pdf. Kept in case a slide wants it back.
def fig_collinearity_contours():
    """RSS contours for two nearly identical features: a long flat valley."""
    beta_true = np.array([1.0, 1.0])
    fig, axes = plt.subplots(1, 2, figsize=FULL)
    for ax, rho in zip(axes, [0.0, 0.995]):
        X = _collinear_design(rho, seed=3)
        rng = np.random.default_rng(11)
        y = X @ beta_true + 0.5 * rng.normal(size=len(X))
        best, *_ = np.linalg.lstsq(X, y, rcond=None)
        g = np.linspace(-3.5, 5.5, 200)
        G0, G1 = np.meshgrid(g, g)
        R = ((y[:, None, None] - G0[None] * X[:, 0][:, None, None]
              - G1[None] * X[:, 1][:, None, None]) ** 2).mean(axis=0)
        ax.contour(G0, G1, R, levels=np.geomspace(R.min() + 1e-3, R.max(), 18),
                   colors=RULE, linewidths=0.65)
        ax.plot(*best, marker="*", ms=13, color=MORANDI_EDGE[MORANDI_CLAY], zorder=5)
        lam = np.linalg.eigvalsh(X.T @ X / len(X))
        ax.set_aspect("equal")
        ax.set_xlabel(r"$\beta_1$")
        ax.set_ylabel(r"$\beta_2$")
        ax.set_title(rf"$\Cor(x_1,x_2)={rho:.3f}$,   $\kappa={lam.max()/lam.min():.0f}$"
                     .replace(r"\Cor", r"\mathrm{corr}"), fontsize=8.4)
    fig.tight_layout(pad=0.4)
    return save(fig, OUT / "collinearity-contours.pdf", notes_svg("collinearity-contours"))

fig_collinearity_contours()

In [ ]:
def fig_collinearity_simulation():
    """Refit on fresh noise 200 times; watch the coefficients scatter."""
    beta_true = np.array([1.0, 1.0])
    fig, axes = plt.subplots(1, 2, figsize=FULL)
    for ax, rho in zip(axes, [0.0, 0.995]):
        X = _collinear_design(rho, seed=3)
        rng = np.random.default_rng(7)
        ests, preds = [], []
        for _ in range(300):
            y = X @ beta_true + 0.5 * rng.normal(size=len(X))
            b, *_ = np.linalg.lstsq(X, y, rcond=None)
            ests.append(b)
            preds.append(X @ b)
        ests = np.array(ests)
        spread_pred = np.std(np.array(preds), axis=0).mean()
        ax.scatter(ests[:, 0], ests[:, 1], s=9, color=MORANDI_BLUE, alpha=0.5, linewidths=0)
        ax.plot(*beta_true, marker="*", ms=14, color=MORANDI_EDGE[MORANDI_CLAY], zorder=5)
        ax.set_xlim(beta_true[0] - 1.4, beta_true[0] + 1.4)
        ax.set_ylim(beta_true[1] - 1.4, beta_true[1] + 1.4)
        ax.set_aspect("equal")
        ax.set_xlabel(r"$\widehat\beta_1$")
        ax.set_ylabel(r"$\widehat\beta_2$")
        ax.set_title(rf"corr $={rho:.3f}$:  sd$(\widehat\beta_1)={ests[:,0].std():.2f}$,"
                     "\n" rf"but sd of fitted values $={spread_pred:.2f}$", fontsize=8.2)
    fig.tight_layout(pad=0.4)
    return save(fig, OUT / "collinearity-simulation.pdf",
                notes_svg("collinearity-simulation"))

fig_collinearity_simulation()

In [ ]:
# --- 12. Heavy-tailed noise breaks least squares ----------------------------
def fig_heavy_tails():
    """Contaminating a handful of markets is enough to move the fit.

    Only the least-squares line is shown: the remedy (Huber, absolute loss) is
    left to the homework, so the slide states the failure without the fix.
    """
    y = sales.copy()
    idx = np.argsort(tv)[:12]
    y_bad = y.copy()
    y_bad[idx[:6]] += 28.0

    X = np.c_[ones, tv]
    grid = np.linspace(tv.min(), tv.max(), 2)
    fig, axes = plt.subplots(1, 2, figsize=FULL, sharey=True)
    for ax, (yv, title, colour) in zip(
        axes,
        [(y, "original data", BLUE), (y_bad, "six of 200 markets contaminated", RED)],
    ):
        b_ols, *_ = np.linalg.lstsq(X, yv, rcond=None)
        ax.scatter(tv, yv, s=11, color=MORANDI_BLUE, alpha=0.5, linewidths=0)
        ax.plot(grid, b_ols[0] + b_ols[1] * grid, color=colour, lw=2.2)
        ax.set_xlabel("TV budget")
        ax.set_title(f"{title}: slope {b_ols[1]:.4f}", fontsize=8.6)
    axes[0].set_ylabel("sales")
    fig.tight_layout(pad=0.4)
    return save(fig, OUT / "heavy-tails.pdf", notes_svg("heavy-tails"))

fig_heavy_tails()

In [ ]:
# --- 13. Underfitting and overfitting with polynomial features -------------
def fig_overfit_underfit():
    """Polynomial fits of sales on TV, on a small training subset."""
    rng = np.random.default_rng(1)
    perm = rng.permutation(len(tv))
    tr, te = perm[:25], perm[25:]
    xg = np.linspace(tv.min(), tv.max(), 400)

    fig = plt.figure(figsize=FULL)
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.15])
    for k, deg in enumerate([1, 12]):
        ax = fig.add_subplot(gs[0, k])
        c = np.polyfit(tv[tr], sales[tr], deg)
        ax.scatter(tv[tr], sales[tr], s=16, color=MORANDI_BLUE, zorder=3, linewidths=0)
        ax.plot(xg, np.polyval(c, xg), color=MORANDI_EDGE[MORANDI_CLAY], lw=1.9)
        ax.set_ylim(sales.min() - 3, sales.max() + 3)
        ax.set_xlabel("TV budget")
        ax.set_title(f"degree {deg}: {'underfits' if deg == 1 else 'overfits'}",
                     fontsize=8.4)
        if k == 0:
            ax.set_ylabel("sales")

    ax = fig.add_subplot(gs[0, 2])
    degs = np.arange(1, 15)
    tr_err, te_err = [], []
    for d in degs:
        c = np.polyfit(tv[tr], sales[tr], d)
        tr_err.append(np.mean((sales[tr] - np.polyval(c, tv[tr])) ** 2))
        te_err.append(np.mean((sales[te] - np.polyval(c, tv[te])) ** 2))
    ax.semilogy(degs, tr_err, "-o", ms=3.5, color=BLUE, label="training")
    ax.semilogy(degs, te_err, "-o", ms=3.5, color=RED, label="held out")
    ax.set_xlabel("polynomial degree")
    ax.set_ylabel("mean squared error")
    ax.legend(loc="upper center", fontsize=7.4)
    fig.tight_layout(pad=0.4)
    return save(fig, OUT / "overfit-underfit.pdf", notes_svg("overfit-underfit"))

fig_overfit_underfit()

In [ ]:
# --- 10b. The tangent-line characterisation, in one variable ---------------
def fig_convexity_tangent():
    """A convex curve with two tangent lines, both lying below it.

    Lecture 03 states the first-order condition rather than deriving it, so the
    picture has to carry the intuition.
    """
    t = np.linspace(-2.6, 2.6, 500)
    f = lambda z: 0.55 * z**2 + 0.5
    g = lambda z: 1.10 * z

    fig, ax = plt.subplots(figsize=FULL)
    ax.plot(t, f(t), color=MORANDI_EDGE[MORANDI_BLUE], lw=2.4, zorder=3, label=r"$L(\beta)$")
    for a, col in [(-1.6, RED), (1.1, GREEN)]:
        ax.plot(t, f(a) + g(a) * (t - a), color=col, lw=1.6, ls=(0, (5, 3)), zorder=2)
        ax.plot([a], [f(a)], "o", ms=6, color=col, zorder=4)
        ax.annotate(rf"tangent at $\beta={a:g}$", xy=(a, f(a)),
                    xytext=(0, -34 if a < 0 else -30), textcoords="offset points",
                    ha="center", color=col, fontsize=8.4)
    ax.set_ylim(-1.6, 4.6)
    ax.set_xlabel(r"$\beta$")
    ax.set_ylabel(r"$L(\beta)$")
    ax.set_yticks([])
    ax.legend(loc="upper center", fontsize=8.5)
    ax.set_title("a convex function never dips below any of its tangents", fontsize=8.8)
    fig.tight_layout(pad=0.3)
    return save(fig, OUT / "convexity-tangent.pdf", notes_svg("convexity-tangent"))

fig_convexity_tangent()

In [ ]:
# --- 10c. What a stationary point can be, when the function is not convex --
# UNUSED: no deck includes stationary-points.pdf. Kept in case a slide wants it back.
def fig_stationary_points():
    """Left: a nonconvex objective, with every kind of stationary point.

    Right: a convex one, where the only stationary point is the global minimum.
    This is the picture behind "why did we trust the stationary point?".
    """
    fig, axes = plt.subplots(1, 2, figsize=FULL)

    # A quartic with two minima and one maximum, so every label is honest.
    t = np.linspace(-2.15, 2.15, 900)
    f = t**4 - 3.0 * t**2 + 0.75 * t
    ax = axes[0]
    ax.plot(t, f, color=MORANDI_EDGE[MORANDI_BLUE], lw=2.2, zorder=3)

    # Exact stationary points: f' = 4t^3 - 6t + 0.75.
    roots = np.sort(np.roots([4.0, 0.0, -6.0, 0.75]).real)
    fr = roots**4 - 3.0 * roots**2 + 0.75 * roots
    curv = 12.0 * roots**2 - 6.0
    gmin = int(np.argmin(np.where(curv > 0, fr, np.inf)))
    for k, (tk, fk, ck) in enumerate(zip(roots, fr, curv)):
        if ck > 0:
            label = "global minimum" if k == gmin else "local minimum"
            colour, dy = (GREEN, 14) if k == gmin else (RED, 14)
        else:
            label, colour, dy = "local maximum", RED, -22
        ax.plot(tk, fk, "o", ms=6, color=colour, zorder=4)
        ax.annotate(label, xy=(tk, fk), xytext=(0, dy), textcoords="offset points",
                    ha="center", color=colour, fontsize=8)
    ax.set_ylim(fr.min() - 1.1, f.max() + 1.4)
    ax.set_title("not convex: a flat point can be anything", fontsize=8.6)

    ax = axes[1]
    tc = np.linspace(-2.4, 2.4, 400)
    fc = 0.7 * tc**2 + 0.4
    ax.plot(tc, fc, color=MORANDI_EDGE[MORANDI_BLUE], lw=2.2, zorder=3)
    ax.plot([0], [0.4], "*", ms=15, color=MORANDI_EDGE[MORANDI_SAGE], zorder=4)
    ax.annotate("the only flat point,\nand it is the global minimum",
                xy=(0, 0.4), xytext=(0, 22), textcoords="offset points",
                ha="center", color=GREEN, fontsize=8)
    ax.set_title("convex: a flat point is the minimum", fontsize=8.6)

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel(r"$\beta$")
    fig.tight_layout(pad=0.4)
    return save(fig, OUT / "stationary-points.pdf", notes_svg("stationary-points"))

fig_stationary_points()

In [ ]:
# --- 10e. The chord definition of convexity, in one variable ---------------
def fig_convexity_chord():
    """A chord between two points of a convex curve stays above the curve."""
    t = np.linspace(-2.3, 2.3, 500)
    f = lambda z: 0.55 * z**2 + 0.5
    a, b = -1.7, 1.35
    lam = 0.62
    mid = lam * a + (1 - lam) * b

    fig, ax = plt.subplots(figsize=FULL)
    ax.plot(t, f(t), color=MORANDI_EDGE[MORANDI_BLUE], lw=2.4, zorder=3)
    ax.plot([a, b], [f(a), f(b)], color=MORANDI_EDGE[MORANDI_CLAY], lw=1.9, zorder=2)
    ax.plot([a, b], [f(a), f(b)], "o", ms=6, color=MORANDI_EDGE[MORANDI_CLAY], zorder=4)
    chord_at_mid = lam * f(a) + (1 - lam) * f(b)
    ax.plot([mid, mid], [f(mid), chord_at_mid], color=MORANDI_EDGE[MORANDI_SAGE], lw=1.6,
            ls=(0, (3, 2)), zorder=4)
    ax.plot([mid], [chord_at_mid], "o", ms=5, color=MORANDI_EDGE[MORANDI_CLAY], zorder=5)
    ax.plot([mid], [f(mid)], "o", ms=5, color=MORANDI_EDGE[MORANDI_BLUE], zorder=5)
    ax.annotate(r"$\lambda L(\beta)+(1-\lambda)L(\beta')$",
                xy=(mid, chord_at_mid), xytext=(10, 12), textcoords="offset points",
                color=RED, fontsize=8.6)
    ax.annotate(r"$L(\lambda\beta+(1-\lambda)\beta')$",
                xy=(mid, f(mid)), xytext=(14, -4), textcoords="offset points",
                color=BLUE, fontsize=8.6)
    for x, lab in [(a, r"$\beta$"), (b, r"$\beta'$")]:
        ax.annotate(lab, xy=(x, f(x)), xytext=(0, 10), textcoords="offset points",
                    ha="center", color=RED, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(r"$\beta$")
    ax.set_ylim(0.25, 3.6)
    fig.tight_layout(pad=0.3)
    return save(fig, OUT / "convexity-chord.pdf", notes_svg("convexity-chord"))

fig_convexity_chord()